# 01 - Data Exploration (Khám phá dữ liệu)

**Mục đích:** hiểu dữ liệu gốc trước khi làm sạch. Notebook này chỉ **QUAN SÁT**.

**Nguyên tắc:**
- Không xóa, không sửa, không ghi file nào. Mọi quyết định xử lý dữ liệu để dành cho notebook 02.
- Mỗi mục gồm: *câu hỏi kiểm tra* → *code* → chỗ để bạn ghi *nhận xét*.

## 0. Chuẩn bị

In [1]:
import sys
from pathlib import Path

import pandas as pd

# Tự động nhận diện PROJECT_ROOT an toàn dù kernel chạy từ notebooks/ hay từ thư mục gốc
current_dir = Path.cwd().resolve()
PROJECT_ROOT = current_dir.parent if current_dir.name == "notebooks" else current_dir
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data.load_data import load_raw_data

# Hiển thị số thực gọn hơn (2 chữ số thập phân), tránh in dạng khoa học 1e+04 khó đọc
pd.options.display.float_format = "{:.2f}".format


## 1. Nạp dữ liệu, kiểm tra shape / columns / dtypes

**Câu hỏi:** dữ liệu có bao nhiêu dòng, cột? Mỗi cột có kiểu dữ liệu gì và có đúng kiểu mong đợi không?

In [2]:
df = load_raw_data(r"C:\Users\chaud\Downloads\online-retail-customer-analytics-main\online-retail-customer-analytics-main\data\raw\Online Retail .xlsx")   # có thể mất vài chục giây vì đọc file Excel

print("Shape:", df.shape)
print("Columns:", list(df.columns))
print()
print(df.dtypes)
df.head()

Shape: (541909, 8)
Columns: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

InvoiceNo                 str
StockCode                 str
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID            float64
Country                   str
dtype: object


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.00,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.00,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.00,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.00,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.00,United Kingdom


**Nhận xét của bạn:** (ghi lại: bao nhiêu dòng? `InvoiceDate` có phải datetime không? `CustomerID` là kiểu gì, vì sao?)

## 2. Missing values

**Câu hỏi:** cột nào bị thiếu, thiếu bao nhiêu %?

In [3]:
missing = pd.DataFrame({
    "so_dong_thieu": df.isna().sum(),
    "ty_le_%": df.isna().mean() * 100,
})
missing[missing["so_dong_thieu"] > 0]

,so_dong_thieu,ty_le_%
Description,1454,0.27
CustomerID,135080,24.93


**Nhận xét của bạn:** (cột nào thiếu nhiều nhất? Tỷ lệ bao nhiêu? Điều này ảnh hưởng thế nào đến RFM, vốn cần `CustomerID`?)

### 2b. Dòng thiếu `Description` có đặc điểm gì?

**Câu hỏi:** những dòng không có mô tả sản phẩm là dòng bình thường hay dòng đặc biệt? Nhìn `UnitPrice` và `CustomerID` của chúng.

In [4]:
no_desc = df[df["Description"].isna()]

print("Số dòng thiếu Description:", len(no_desc))
print("Trong đó UnitPrice == 0     :", (no_desc["UnitPrice"] == 0).sum())
print("Trong đó thiếu CustomerID   :", no_desc["CustomerID"].isna().sum())
no_desc.head()

Số dòng thiếu Description: 1454
Trong đó UnitPrice == 0     : 1454
Trong đó thiếu CustomerID   : 1454


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.00,NaN,United Kingdom
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.00,NaN,United Kingdom
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.00,NaN,United Kingdom
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.00,NaN,United Kingdom
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.00,NaN,United Kingdom


## 3. Duplicate (dòng trùng lặp)

**Câu hỏi:** có bao nhiêu dòng giống hệt nhau ở cả 8 cột? Chúng có vẻ là lỗi nhập liệu hay là mua trùng hợp lệ?

In [5]:
n_dup = df.duplicated().sum()
print("Số dòng trùng (không tính bản đầu tiên):", n_dup)
print("Tỷ lệ: {:.2f}%".format(n_dup / len(df) * 100))

# keep=False: hiện TẤT CẢ các dòng thuộc nhóm trùng (kể cả bản gốc) để so sánh
dup_rows = df[df.duplicated(keep=False)].sort_values(["InvoiceNo", "StockCode"])
dup_rows.head(8)

Số dòng trùng (không tính bản đầu tiên): 5268
Tỷ lệ: 0.97%


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.00,United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.00,United Kingdom
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.00,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.00,United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.00,United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.00,United Kingdom
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.00,United Kingdom
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.00,United Kingdom


**Nhận xét của bạn:** (các dòng trùng thường thuộc cùng một invoice và cùng thời điểm. Bạn nghĩ đây là lỗi hay khách mua 2 lần cùng món? Dữ liệu **không đủ thông tin để khẳng định**, nên quyết định xóa hay giữ phải được giải thích ở notebook 02.)

## 4. Quantity

**Câu hỏi:** `Quantity` có giá trị âm hoặc bằng 0 không? Có giá trị cực đoan không?

In [6]:
print("Quantity < 0 :", (df["Quantity"] < 0).sum())
print("Quantity == 0:", (df["Quantity"] == 0).sum())
print()
print(df["Quantity"].describe())
print()
print("5 giá trị nhỏ nhất:", df["Quantity"].nsmallest(5).tolist())
print("5 giá trị lớn nhất:", df["Quantity"].nlargest(5).tolist())

Quantity < 0 : 10624
Quantity == 0: 0

count   541909.00
mean         9.55
std        218.08
min     -80995.00
25%          1.00
50%          3.00
75%         10.00
max      80995.00
Name: Quantity, dtype: float64

5 giá trị nhỏ nhất: [-80995, -74215, -9600, -9600, -9360]
5 giá trị lớn nhất: [80995, 74215, 12540, 5568, 4800]


**Nhận xét của bạn:** (Quantity âm nghĩa là gì? Median so với max cho thấy điều gì về độ lệch của phân phối?)

## 5. UnitPrice

**Câu hỏi:** `UnitPrice` có giá trị âm hoặc bằng 0 không? Những dòng đó là gì?

In [7]:
print("UnitPrice < 0 :", (df["UnitPrice"] < 0).sum())
print("UnitPrice == 0:", (df["UnitPrice"] == 0).sum())
print()

print("--- Các dòng UnitPrice < 0 ---")
display(df[df["UnitPrice"] < 0])

print("--- Các dòng UnitPrice == 0: có CustomerID không? ---")
zero_price = df[df["UnitPrice"] == 0]
print("Có CustomerID  :", zero_price["CustomerID"].notna().sum())
print("Thiếu CustomerID:", zero_price["CustomerID"].isna().sum())

UnitPrice < 0 : 2
UnitPrice == 0: 2515

--- Các dòng UnitPrice < 0 ---


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


--- Các dòng UnitPrice == 0: có CustomerID không? ---
Có CustomerID  : 40
Thiếu CustomerID: 2475


**Nhận xét của bạn:** (giá âm xuất hiện ở đâu, `Description` là gì? Đây có phải giao dịch bán hàng thông thường không?)

## 6. InvoiceDate

**Câu hỏi:** dữ liệu trải dài trong khoảng thời gian nào? Có đủ các ngày trong tuần không? (Quan trọng cho câu hỏi weekday vs weekend ở Phase 6.)

In [8]:
print("Ngày sớm nhất :", df["InvoiceDate"].min())
print("Ngày muộn nhất:", df["InvoiceDate"].max())
print("Số ngày có giao dịch:", df["InvoiceDate"].dt.date.nunique())
print()

# Số dòng giao dịch theo thứ trong tuần
print(df["InvoiceDate"].dt.day_name().value_counts())

# Tháng cuối cùng có đầy đủ không?
last_month = df[df["InvoiceDate"].dt.to_period("M") == "2011-12"]
print("\nDữ liệu tháng 12/2011 kết thúc ngày:", last_month["InvoiceDate"].max().date())

Ngày sớm nhất : 2010-12-01 08:26:00
Ngày muộn nhất: 2011-12-09 12:50:00
Số ngày có giao dịch: 305

InvoiceDate
Thursday     103857
Tuesday      101808
Monday        95111
Wednesday     94565
Friday        82193
Sunday        64375
Name: count, dtype: int64

Dữ liệu tháng 12/2011 kết thúc ngày: 2011-12-09


**Nhận xét của bạn:** (ngày nào trong tuần **không có** dữ liệu? Tháng 12/2011 có đủ cả tháng không? Điều này ảnh hưởng thế nào khi so sánh doanh thu theo tháng?)

## 7. InvoiceNo và đơn hủy

**Câu hỏi:** `InvoiceNo` có những dạng nào? Đơn hủy (bắt đầu bằng "C") có khớp với Quantity âm không?

Ở đây mình chỉ tạo một Series tạm để quan sát. Biến `IsCancelled` chính thức sẽ được tạo ở notebook 02.

In [9]:
inv = df["InvoiceNo"]

print("Độ dài InvoiceNo:")
print(inv.str.len().value_counts())
print()
print("Ký tự đầu KHÔNG phải chữ số:")
print(inv[~inv.str[0].str.isdigit()].str[0].value_counts())
print()

# Series tạm, chưa thêm vào df
is_cancelled_tmp = inv.str.startswith("C")

print("Số dòng đơn hủy         :", is_cancelled_tmp.sum())
print("Số invoice hủy (khác nhau):", inv[is_cancelled_tmp].nunique())
print("Tổng số invoice (khác nhau):", inv.nunique())
print()

# So khớp giữa "đơn hủy" và "Quantity âm"
pd.crosstab(
    is_cancelled_tmp.rename("Bat_dau_bang_C"),
    (df["Quantity"] < 0).rename("Quantity_am"),
)

Độ dài InvoiceNo:
InvoiceNo
6    532618
7      9291
Name: count, dtype: int64

Ký tự đầu KHÔNG phải chữ số:
InvoiceNo
C    9288
A       3
Name: count, dtype: int64

Số dòng đơn hủy         : 9288
Số invoice hủy (khác nhau): 3836
Tổng số invoice (khác nhau): 25900



Quantity_am,False,True
Bat_dau_bang_C,,
False,531285,1336
True,0,9288


**Nhận xét của bạn:** (mọi đơn "C" đều có Quantity âm chứ? Ngược lại, có dòng Quantity âm mà **không phải** đơn "C" không? Có bao nhiêu?)

In [10]:
# Các dòng Quantity âm nhưng KHÔNG phải đơn hủy: chúng là gì?
neg_not_cancel = df[(df["Quantity"] < 0) & (~is_cancelled_tmp)]

print("Số dòng:", len(neg_not_cancel))
print("Trong đó UnitPrice == 0    :", (neg_not_cancel["UnitPrice"] == 0).sum())
print("Trong đó thiếu CustomerID  :", neg_not_cancel["CustomerID"].isna().sum())
neg_not_cancel.head()

Số dòng: 1336
Trong đó UnitPrice == 0    : 1336
Trong đó thiếu CustomerID  : 1336


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.00,NaN,United Kingdom
4347,536764,84952C,NaN,-38,2010-12-02 14:42:00,0.00,NaN,United Kingdom
7188,536996,22712,NaN,-20,2010-12-03 15:30:00,0.00,NaN,United Kingdom
7189,536997,22028,NaN,-20,2010-12-03 15:30:00,0.00,NaN,United Kingdom
7190,536998,85067,NaN,-6,2010-12-03 15:30:00,0.00,NaN,United Kingdom


## 8. StockCode không phải sản phẩm

**Câu hỏi:** có mã `StockCode` nào chỉ gồm chữ cái (không giống mã hàng thông thường như `85123A`)? Chúng là gì?

In [11]:
# Mã chỉ gồm chữ cái, ví dụ POST, DOT, M, AMAZONFEE ...
letters_only = df["StockCode"].str.match(r"^[A-Za-z]+$")

print(df.loc[letters_only, "StockCode"].value_counts())
print()

# Xem mỗi mã tương ứng với Description nào
(df[letters_only]
   .groupby("StockCode")["Description"]
   .agg(lambda s: s.dropna().iloc[0] if s.notna().any() else None))

StockCode
POST         1256
DOT           710
M             571
D              77
S              63
AMAZONFEE      34
CRUK           16
DCGSSGIRL      13
DCGSSBOY       11
PADS            4
B               3
m               1
Name: count, dtype: int64



StockCode
AMAZONFEE                    AMAZON FEE
B                       Adjust bad debt
CRUK                    CRUK Commission
D                              Discount
DCGSSBOY                 BOYS PARTY BAG
DCGSSGIRL               GIRLS PARTY BAG
DOT                      DOTCOM POSTAGE
M                                Manual
PADS         PADS TO MATCH ALL CUSHIONS
POST                            POSTAGE
S                               SAMPLES
m                                Manual
Name: Description, dtype: str

**Nhận xét của bạn:** (những mã này là phí vận chuyển, phí hệ thống, điều chỉnh thủ công... chứ không phải hàng hóa. Có nên tính vào 'Revenue theo sản phẩm' không? Quyết định ở notebook 02.)

## 9. CustomerID

**Câu hỏi:** có bao nhiêu khách hàng khác nhau? Dòng thiếu `CustomerID` có đặc điểm gì?

In [12]:
print("Số khách hàng khác nhau:", df["CustomerID"].nunique())
print("Số dòng thiếu CustomerID:", df["CustomerID"].isna().sum())
print("Số invoice thiếu CustomerID:", df.loc[df["CustomerID"].isna(), "InvoiceNo"].nunique())
print()

# CustomerID có phải số nguyên (dù đang lưu dạng float) không?
print("CustomerID có phần thập phân:", (df["CustomerID"].dropna() % 1 != 0).sum(), "dòng")

# Một invoice có thể thuộc nhiều khách hàng không?
cust_per_invoice = df.groupby("InvoiceNo")["CustomerID"].nunique()
print("Invoice có > 1 CustomerID:", (cust_per_invoice > 1).sum())

# Thiếu CustomerID có liên quan tới đơn hủy / giá 0 không?
missing_cust = df["CustomerID"].isna()
print()
print("Thiếu CustomerID & đơn hủy   :", (missing_cust & is_cancelled_tmp).sum())
print("Thiếu CustomerID & UnitPrice=0:", (missing_cust & (df["UnitPrice"] == 0)).sum())

Số khách hàng khác nhau: 4372
Số dòng thiếu CustomerID: 135080
Số invoice thiếu CustomerID: 3710

CustomerID có phần thập phân: 0 dòng
Invoice có > 1 CustomerID: 0

Thiếu CustomerID & đơn hủy   : 383
Thiếu CustomerID & UnitPrice=0: 2475


**Nhận xét của bạn:** (một invoice có thuộc nhiều khách hàng không? Điều đó cho phép gán invoice → khách hàng một cách nhất quán không?)

## 10. Country

**Câu hỏi:** dữ liệu phân bố ra sao giữa các quốc gia? Có giá trị bất thường không?

In [13]:
print("Số quốc gia:", df["Country"].nunique())
country_share = df["Country"].value_counts()
print(country_share.head(8))
print("...")
print(country_share.tail(4))

print()
print("Tỷ lệ dòng thuộc United Kingdom: {:.1f}%".format(
    (df["Country"] == "United Kingdom").mean() * 100))
print("Country = 'Unspecified'      :", (df["Country"] == "Unspecified").sum())
print("Country = 'European Community':", (df["Country"] == "European Community").sum())

Số quốc gia: 38
Country
United Kingdom    495478
Germany             9495
France              8557
EIRE                8196
Spain               2533
Netherlands         2371
Belgium             2069
Switzerland         2002
Name: count, dtype: int64
...
Country
Brazil            32
Czech Republic    30
Bahrain           19
Saudi Arabia      10
Name: count, dtype: int64

Tỷ lệ dòng thuộc United Kingdom: 91.4%
Country = 'Unspecified'      : 446
Country = 'European Community': 61


**Nhận xét của bạn:** (dữ liệu lệch mạnh về nước nào? Điều này ảnh hưởng thế nào khi so sánh các quốc gia sau này, ví dụ nhóm nước có ít dòng?)

## 11. Tổng hợp phát hiện và việc cần quyết định ở notebook 02

| # | Phát hiện | Con số | Câu hỏi cho notebook 02 |
|---|---|---|---|
| 1 | Kích thước | 541.909 dòng, 8 cột | - |
| 2 | Thiếu `CustomerID` | 135.080 dòng (24,93%) | Giữ cho phân tích doanh thu, loại khỏi RFM? |
| 3 | Thiếu `Description` | 1.454 dòng, toàn bộ có `UnitPrice = 0` và không có `CustomerID` | Có phải giao dịch bán hàng không? |
| 4 | Dòng trùng hoàn toàn | 5.268 dòng | Lỗi nhập liệu hay mua trùng hợp lệ? Loại hay giữ? |
| 5 | Đơn hủy (bắt đầu bằng "C") | 9.288 dòng, 3.836 invoice; **tất cả** có Quantity âm | Tách riêng bằng `IsCancelled`, không xóa |
| 6 | Quantity âm **không phải** đơn hủy | 1.336 dòng, đều có `UnitPrice = 0` và thiếu `CustomerID` | Điều chỉnh kho, không phải bán hàng? |
| 7 | `UnitPrice = 0` | 2.515 dòng (40 dòng có `CustomerID`) | Loại khỏi phân tích doanh thu? |
| 8 | `UnitPrice < 0` | 2 dòng, `Adjust bad debt`, invoice bắt đầu bằng "A" | Không phải bán hàng |
| 9 | `StockCode` không phải hàng hóa | POST, DOT, M, D, S, AMAZONFEE, CRUK, ... | Loại khỏi "Revenue theo sản phẩm"? |
| 10 | Giá trị cực đoan | Quantity ±80.995; UnitPrice tới 38.970 (đa số là `AMAZONFEE`, `Manual`, `POSTAGE`) | Xử lý outlier thế nào? |
| 11 | Khoảng thời gian | 01/12/2010 - 09/12/2011; tháng 12/2011 chỉ có đến ngày 09 | Lưu ý khi so sánh doanh thu theo tháng |
| 12 | Thứ trong tuần | Không có giao dịch nào vào thứ Bảy | "Weekend" thực chất chỉ còn Chủ nhật: ảnh hưởng câu hỏi kiểm định |
| 13 | Quốc gia | 38 nước; Anh chiếm khoảng 91% số dòng | Nhóm nước ít dữ liệu cần gộp khi kiểm định Chi-square |

> Đây mới là **quan sát**. Chưa có quyết định xóa hay giữ nào được đưa ra.